# 🚀 HUẤN LUYỆN CHUYÊN SÂU MÔ HÌNH YOLOPv2 ĐA MIỀN TRÊN GOOGLE COLAB
### ĐỀ TÀI: HỆ THỐNG PHÁT HIỆN LÀN ĐƯỜNG & PHƯƠNG TIỆN GIAO THÔNG
**Nhóm 1 — Lớp 24DHTT02 | HK7 (2026–2027)**

---
### 📋 HƯỚNG DẪN 2 BƯỚC ĐƠN GIẢN (CHỈ CẦN KÉO THẢ & BẤM RUN ALL):
1. **Bước 1**: Nhìn sang cột bên trái của Google Colab, bấm vào biểu tượng 📁 **Files**.
2. **Bước 2**: Kéo thả file `colab_training_bundle.zip` (chỉ 8 MB từ máy tính của bạn) vào đó.
3. **Bước 3**: Chọn menu **Runtime (Thời gian chạy) ➔ Run all (Chạy tất cả)**.

> ⚡ **Hệ thống sẽ tự động 100%**:
> - Giải nén dữ liệu và mã nguồn chuẩn hóa
> - Tự động tải file trọng số gốc `yolopv2.pt` (156 MB) từ GitHub Release siêu tốc
> - Huấn luyện chuyên sâu 20 epochs trên GPU Tesla T4 (chỉ khoảng 6–8 phút)
> - Vẽ biểu đồ Loss/Metrics và **TỰ ĐỘNG TẢI FILE TRỌNG SỐ VỀ MÁY CỦA BẠN** khi kết thúc!

## 1. Kiểm tra cấu hình phần cứng GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Kết nối Google Drive & Thiết lập Môi trường làm việc

In [ ]:
import os
from pathlib import Path

# 1. Giải nén gói mã nguồn & dữ liệu nếu người dùng upload file colab_training_bundle.zip
if Path('/content/colab_training_bundle.zip').exists():
    print('📦 Tìm thấy colab_training_bundle.zip! Đang giải nén...')
    !unzip -q -o /content/colab_training_bundle.zip -d /content/
    print('✓ Giải nén dữ liệu và mã nguồn thành công!')
elif Path('/content/drive/MyDrive/ChuongTrinhPhatHienLanDuong').exists():
    %cd /content/drive/MyDrive/ChuongTrinhPhatHienLanDuong
    print('✓ Đã kết nối thư mục dự án từ Google Drive!')
else:
    print('ℹ️ Lưu ý: Nếu bạn chưa upload colab_training_bundle.zip, vui lòng kéo thả file đó vào cột Files bên trái.')

# 2. Tự động tải file trọng số yolopv2.pt gốc từ GitHub Release nếu chưa có
!mkdir -p weights
if not Path('weights/yolopv2.pt').exists() or Path('weights/yolopv2.pt').stat().st_size < 100000000:
    print('📥 Đang tự động tải trọng số yolopv2.pt gốc từ GitHub Release...')
    !wget -q --show-progress -O weights/yolopv2.pt https://github.com/CAIC-AD/YOLOPv2/releases/download/V0.0.1/yolopv2.pt
    print('✓ Tải trọng số gốc yolopv2.pt thành công!')
else:
    print('✓ Đã có sẵn file trọng số weights/yolopv2.pt.')


## 3. Cài đặt các thư viện phụ thuộc

In [ ]:
!pip install -q ultralytics torchvision opencv-python matplotlib scipy
!pip install -q -r requirements.txt

## 4. Xác nhận Tập Dữ Liệu Đa Miền (Dataset Verification)

In [ ]:
from pathlib import Path
import yaml

data_dir = Path('data/dataset')
print("Kiểm tra các thư mục dữ liệu:")
for sp in ['train', 'val', 'test']:
    n_img = len(list((data_dir / sp / 'images').glob('*.*')))
    n_lbl = len(list((data_dir / sp / 'labels').glob('*.txt')))
    n_da = len(list((data_dir / sp / 'drivable').glob('*.png')))
    n_ll = len(list((data_dir / sp / 'lane').glob('*.png')))
    print(f"  • {sp.upper():5s}: {n_img} ảnh, {n_lbl} nhãn box, {n_da} mask lòng đường, {n_ll} mask vạch làn")

## 5. Tiến hành Huấn Luyện 2 Pha (Two-Phase Fine-Tuning)
- **Pha 1 (15 Epochs)**: Đóng băng E-ELAN Backbone, chỉ huấn luyện 3 Heads đầu ra với learning rate $\eta = 10^{-4}$ (AdamW).
- **Pha 2 (35 Epochs)**: Mở khóa 100% mạng nơ-ron, giảm tốc độ học theo Cosine Annealing ($\eta: 10^{-4} \to 10^{-6}$).
- **Hàm mất mát**: Tái cân bằng đa nhiệm với $\lambda_{\text{lane}} = 2.5$ (Focal + Dice Loss), $\lambda_{\text{da}} = 1.0$, $\lambda_{\text{det}} = 1.0$.

In [ ]:
!python training/train_yolopv2.py \
    --weights weights/yolopv2.pt \
    --data data/dataset \
    --epochs-p1 5 \
    --epochs-p2 15 \
    --batch-size 8 \
    --lr 0.0001 \
    --device cuda:0 \
    --output-dir runs/train_colab


## 6. Hiển thị Biểu đồ Quá trình Hội tụ Loss & Metrics

In [ ]:
from IPython.display import Image, display

loss_chart = Path('runs/train_colab/training_loss_curves.png')
if loss_chart.exists():
    print("--- BIỂU ĐỒ HỘI TỤ LOSS ĐA NHIỆM ---")
    display(Image(str(loss_chart)))

metrics_chart = Path('runs/train_colab/metrics_curves.png')
if metrics_chart.exists():
    print("--- BIỂU ĐỒ CHỈ SỐ PHÂN ĐOẠN IoU ---")
    display(Image(str(metrics_chart)))

## 7. Đánh giá Trực quan Kết quả Suy luận trên Ảnh Giao thông Việt Nam & Quốc tế

In [ ]:
import cv2
import matplotlib.pyplot as plt
from lane_detection.yolopv2_detector import YOLOPv2Detector

# Tải mô hình đã huấn luyện tinh chỉnh
detector = YOLOPv2Detector(
    duong_dan_weights='weights/yolopv2_multidomain.pt',
    device='cuda:0' if torch.cuda.is_available() else 'cpu'
)

# Thử nghiệm trên ảnh nội thành TP.HCM
test_img_path = 'data/images/Giao-Thong-Tphcm-10.jpg'
img_bgr = cv2.imread(test_img_path)
ket_qua = detector.phat_hien(img_bgr)

print(f"Số phương tiện phát hiện: {ket_qua['so_luong_xe']}")
print(f"Thời gian xử lý: {ket_qua['thoi_gian_ms']:.1f} ms")

# Vẽ kết quả
vis = img_bgr.copy()
da_mask = ket_qua['mat_na_duong']
ll_mask = ket_qua['mat_na_lan']
vis[da_mask > 0] = cv2.addWeighted(vis, 0.65, np.full_like(vis, (230, 160, 50)), 0.35, 0)[da_mask > 0]
vis[ll_mask > 0] = [0, 0, 255]

for box in ket_qua['hop_xe']:
    x1, y1, x2, y2 = map(int, box[:4])
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 2)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('Kết quả suy luận mô hình sau huấn luyện trên Colab', fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

## 8. Tải về File Trọng số Tối ưu để Tích hợp vào Ứng dụng Desktop

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

best_colab_weight = Path('runs/train_colab/yolopv2_multidomain_best.pt')
final_weight = Path('weights/yolopv2_multidomain.pt')

if best_colab_weight.exists():
    shutil.copyfile(str(best_colab_weight), str(final_weight))
    print(f'✓ Đã chuẩn bị file trọng số tốt nhất ({round(final_weight.stat().st_size / (1024*1024), 1)} MB).')
    print('🚀 Đang tự động tải file yolopv2_multidomain.pt về máy tính của bạn...')
    files.download(str(final_weight))
elif final_weight.exists():
    files.download(str(final_weight))
    print('Đã tải về file trọng số YOLOPv2 hoàn thiện!')
else:
    print('⚠️ Không tìm thấy file trọng số đã huấn luyện.')
